# Input

In [ ]:
import os
import sys
from pathlib import Path

# Thread limits should be set before importing torch / numpy-heavy libraries
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

# Make local ProSST repo importable
PROSST_REPO = Path("/home/kebau102/Masterarbeit/ProSST")

if PROSST_REPO.exists():
    sys.path.insert(0, str(PROSST_REPO))
    print(f"Added ProSST repo to sys.path: {PROSST_REPO}")
else:
    print(f"[WARNING] ProSST repo not found: {PROSST_REPO}")

print("Environment setup done.")

Added ProSST repo to sys.path: /home/kebau102/Masterarbeit/ProSST
Environment setup done.


In [ ]:
import re
from dataclasses import dataclass, field
from typing import Iterable, Optional

import pandas as pd
from scipy.stats import spearmanr
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

from prosst.structure.get_sst_seq import SSTPredictor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Standard imports loaded.
Torch loaded.
Transformers loaded.
ProSST imports loaded.
Using device: cuda


# Path

In [ ]:
DATA_DIR = Path("../../data")
PDB_DIR = DATA_DIR / "pdb"
PETASE_PDB_DIR = DATA_DIR / "PETase_PDB"

OUT_DIR = Path("prosst_zero_shot_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "AI4Protein/ProSST-2048"
STRUCTURE_VOCAB_SIZE = 2048

OFFSET_SEARCH = 200
STRICT_WT_MATCH = False

ALPHA_AMYLASE_PDB = PDB_DIR / "alpha_amylase_wt.pdb"

print(f"Output directory: {OUT_DIR}")
print(f"Alpha-Amylase PDB: {ALPHA_AMYLASE_PDB}")

Output directory: /home/kebau102/Masterarbeit/out/prosst
Alpha-Amylase PDB: /home/kebau102/Masterarbeit/data/pdb/alpha_amylase_wt.pdb


In [ ]:
# Dataset Configuration Class
@dataclass
class DatasetConfig:
    name: str
    data_path: Path
    pdb_path: Path
    out_path: Path

    sequence_col: str = "sequence"
    variant_col: str = "variant"
    score_cols: list[str] = field(default_factory=lambda: ["score"])

    sep: Optional[str] = None
    require_num_mutations_col: bool = False
    require_sequence_length_match: bool = True

    offset_search: int = OFFSET_SEARCH
    strict_wt_match: bool = STRICT_WT_MATCH

In [ ]:
DATASETS = [
    DatasetConfig(
        name="UBE4B",
        data_path=DATA_DIR / "ube4b_with_full_sequence.tsv",
        pdb_path=PDB_DIR / "UBE4B.pdb",
        out_path=OUT_DIR / "UBE4B_prosst.tsv",
        sequence_col="sequence",
        variant_col="variant",
        score_cols=["score"],
        sep="\t",
        require_num_mutations_col=True,
        require_sequence_length_match=True,
    ),

    DatasetConfig(
        name="GRB2",
        data_path=DATA_DIR / "grb2_with_full_sequence.tsv",
        pdb_path=PDB_DIR / "GRB2.pdb",
        out_path=OUT_DIR / "GRB2_prosst.tsv",
        sequence_col="sequence",
        variant_col="variant",
        score_cols=["score"],
        sep="\t",
        require_num_mutations_col=True,
        require_sequence_length_match=True,
    ),

    DatasetConfig(
        name="PTEN_activity",
        data_path=DATA_DIR / "pten-activity_with_full_sequence.tsv",
        pdb_path=PDB_DIR / "PTEN.pdb",
        out_path=OUT_DIR / "PTEN_activity_prosst.tsv",
        sequence_col="sequence",
        variant_col="variant",
        score_cols=["score"],
        sep="\t",
        require_num_mutations_col=True,
        require_sequence_length_match=True,
    ),

    DatasetConfig(
        name="PTEN_abundance",
        data_path=DATA_DIR / "pten-abundance_with_full_sequence.tsv",
        pdb_path=PDB_DIR / "PTEN.pdb",
        out_path=OUT_DIR / "PTEN_abundance_prosst.tsv",
        sequence_col="sequence",
        variant_col="variant",
        score_cols=["score"],
        sep="\t",
        require_num_mutations_col=True,
        require_sequence_length_match=True,
    ),

    DatasetConfig(
        name="Alpha-Amylase",
        data_path=DATA_DIR / "Alpha-Amylase with WT(in silico_ Zero Shot).csv",
        pdb_path=ALPHA_AMYLASE_PDB,
        out_path=OUT_DIR / "Alpha_Amylase_prosst.csv",
        sequence_col="sequence",
        variant_col="mutant",
        score_cols=["thermostability", "expression", "specific activity"],
        sep=None,
        require_num_mutations_col=False,
        require_sequence_length_match=False,
    ),

    DatasetConfig(
        name="PETase_WT_1",
        data_path=DATA_DIR / "PETase" / "WT_1.csv",
        pdb_path=PETASE_PDB_DIR / "wt1_short.pdb",
        out_path=OUT_DIR / "PETase_WT_1_prosst.csv",
        sequence_col="sequence",
        variant_col="variant",
        score_cols=["score"],
        sep=None,
        require_num_mutations_col=False,
        require_sequence_length_match=True,
    ),

    DatasetConfig(
        name="PETase_WT_2",
        data_path=DATA_DIR / "PETase" / "WT_2.csv",
        pdb_path=PETASE_PDB_DIR / "wt2_short.pdb",
        out_path=OUT_DIR / "PETase_WT_2_prosst.csv",
        sequence_col="sequence",
        variant_col="variant",
        score_cols=["score"],
        sep=None,
        require_num_mutations_col=False,
        require_sequence_length_match=True,
    ),

    DatasetConfig(
        name="PETase_WT_3",
        data_path=DATA_DIR / "PETase" / "WT_3.csv",
        pdb_path=PETASE_PDB_DIR / "wt3_short.pdb",
        out_path=OUT_DIR / "PETase_WT_3_prosst.csv",
        sequence_col="sequence",
        variant_col="variant",
        score_cols=["score"],
        sep=None,
        require_num_mutations_col=False,
        require_sequence_length_match=True,
    ),
]


for cfg in DATASETS:
    print(f"{cfg.name:16s} | data={cfg.data_path} | pdb={cfg.pdb_path}")

UBE4B            | data=/home/kebau102/Masterarbeit/data/ube4b_with_full_sequence.tsv | pdb=/home/kebau102/Masterarbeit/data/pdb/UBE4B.pdb
GRB2             | data=/home/kebau102/Masterarbeit/data/grb2_with_full_sequence.tsv | pdb=/home/kebau102/Masterarbeit/data/pdb/GRB2.pdb
PTEN_activity    | data=/home/kebau102/Masterarbeit/data/pten-activity_with_full_sequence.tsv | pdb=/home/kebau102/Masterarbeit/data/pdb/PTEN.pdb
PTEN_abundance   | data=/home/kebau102/Masterarbeit/data/pten-abundance_with_full_sequence.tsv | pdb=/home/kebau102/Masterarbeit/data/pdb/PTEN.pdb
Alpha-Amylase    | data=/home/kebau102/Masterarbeit/data/Alpha-Amylase with WT(in silico_ Zero Shot).csv | pdb=/home/kebau102/Masterarbeit/data/pdb/alpha_amylase_wt.pdb
PETase_WT_1      | data=/home/kebau102/Masterarbeit/data/PETase/WT_1.csv | pdb=/home/kebau102/Masterarbeit/data/PETase_PDB/wt1_short.pdb
PETase_WT_2      | data=/home/kebau102/Masterarbeit/data/PETase/WT_2.csv | pdb=/home/kebau102/Masterarbeit/data/PETase_PDB/wt

In [ ]:
# Load ProSST Model
prosst_model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
).to(DEVICE)

prosst_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

prosst_model.eval()

sst_predictor = SSTPredictor(
    structure_vocab_size=STRUCTURE_VOCAB_SIZE
)

print("ProSST model, tokenizer and SST predictor loaded.")

---------- Load Model on cuda ----------
MODEL: 5.90M parameters
ProSST model, tokenizer and SST predictor loaded.


In [ ]:
# Variant Parsing
VAR_SPLIT = re.compile(r"[:;, \t]+")
VARIANT_PATTERN = re.compile(r"^([A-Z])(\d+)([A-Z])$")


def parse_variant(variant: object) -> list[tuple[str, int, str]]:
    """
    Parses variants like A123B.

    Multi-mutants can be separated by:
    ':', ';', ',', spaces or tabs.

    Returns:
        [(wild_type_residue, position_1based, mutant_residue)]
    """
    if pd.isna(variant):
        return []

    text = str(variant).strip().upper()

    if text in {"", "WT", "WILDTYPE", "WILD_TYPE"}:
        return []

    mutations = []

    for token in [p for p in VAR_SPLIT.split(text) if p]:
        match = VARIANT_PATTERN.match(token)

        if match is None:
            raise ValueError(
                f"Unrecognized variant token '{token}' in variant '{variant}'"
            )

        wt = match.group(1)
        pos = int(match.group(2))
        mt = match.group(3)

        mutations.append((wt, pos, mt))

    return mutations


def is_single_mutation(variant: object) -> bool:
    """
    True if the variant contains exactly one mutation.
    """
    try:
        return len(parse_variant(variant)) == 1
    except ValueError:
        return False

In [ ]:
def read_table(cfg: DatasetConfig) -> pd.DataFrame:
    """
    Reads CSV or TSV depending on cfg.sep.
    If cfg.sep is None, pandas tries to infer the separator.
    """
    if not cfg.data_path.exists():
        raise FileNotFoundError(f"Missing input table: {cfg.data_path}")

    if cfg.sep is None:
        return pd.read_csv(cfg.data_path, sep=None, engine="python")

    return pd.read_csv(cfg.data_path, sep=cfg.sep)


def save_table(df: pd.DataFrame, path: Path) -> None:
    """
    Saves as TSV if suffix is .tsv, otherwise CSV.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.suffix.lower() == ".tsv":
        df.to_csv(path, sep="\t", index=False)
    else:
        df.to_csv(path, index=False)


def get_structure_from_pdb(pdb_path: Path) -> tuple[str, list[int]]:
    """
    Extracts amino acid sequence and ProSST structure tokens from a PDB file.
    """
    if not pdb_path.exists():
        raise FileNotFoundError(f"Missing PDB file: {pdb_path}")

    pred = sst_predictor.predict_from_pdb(str(pdb_path))[0]

    pdb_aa_seq = pred["aa_seq"].strip().upper()
    structure_sequence = pred[f"{STRUCTURE_VOCAB_SIZE}_sst_seq"]

    if len(pdb_aa_seq) != len(structure_sequence):
        raise ValueError(
            f"PDB aa_seq length ({len(pdb_aa_seq)}) differs from "
            f"SST length ({len(structure_sequence)}) for {pdb_path}"
        )

    return pdb_aa_seq, structure_sequence


def validate_sequence_length(
    wt_seq: str,
    pdb_seq: str,
    cfg: DatasetConfig,
) -> None:
    """
    Optional check: table WT sequence and PDB sequence should have same length.
    """
    if cfg.require_sequence_length_match and len(wt_seq) != len(pdb_seq):
        raise ValueError(
            f"{cfg.name}: WT sequence length ({len(wt_seq)}) differs from "
            f"PDB sequence length ({len(pdb_seq)})."
        )

In [ ]:
def build_ss_input_ids(structure_sequence: Iterable[int]) -> torch.Tensor:
    """
    ProSST expects:
    - start token: 1
    - structure tokens shifted by +3
    - end token: 2
    """
    structure_sequence_offset = [int(i) + 3 for i in structure_sequence]

    ss_input_ids = torch.tensor(
        [1, *structure_sequence_offset, 2],
        dtype=torch.long,
    ).unsqueeze(0)

    return ss_input_ids


def tokenize_reference_sequence(
    ref_seq: str,
    structure_sequence: Iterable[int],
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Tokenizes the protein sequence and builds corresponding structure input IDs.
    """
    tokenized = prosst_tokenizer(
        [ref_seq],
        return_tensors="pt",
    )

    input_ids = tokenized["input_ids"].to(DEVICE)
    attention_mask = tokenized["attention_mask"].to(DEVICE)

    ss_input_ids = build_ss_input_ids(structure_sequence).to(DEVICE)

    if input_ids.shape[1] != ss_input_ids.shape[1]:
        raise ValueError(
            f"Tokenized sequence length ({input_ids.shape[1]}) differs from "
            f"structure token length ({ss_input_ids.shape[1]})."
        )

    return input_ids, attention_mask, ss_input_ids

In [ ]:
def find_best_offset(
    df: pd.DataFrame,
    ref_seq: str,
    variant_col: str,
    search: int = OFFSET_SEARCH,
) -> tuple[int, int, int]:
    """
    Finds the best offset d such that:

        ref_seq[pos1 + d - 1] == wild-type residue

    This helps when mutation numbering differs from PDB indexing.
    """
    parsed = []

    for variant in df[variant_col].tolist():
        muts = parse_variant(variant)

        if len(muts) == 1:
            parsed.append(muts[0])

    best_offset = 0
    best_hits = -1
    best_total = 0

    n = len(ref_seq)

    for offset in range(-search, search + 1):
        hits = 0
        total = 0

        for wt, pos1, _ in parsed:
            pos_ref = pos1 + offset

            if 1 <= pos_ref <= n:
                total += 1

                if ref_seq[pos_ref - 1].upper() == wt:
                    hits += 1

        if hits > best_hits:
            best_offset = offset
            best_hits = hits
            best_total = total

    return best_offset, best_hits, best_total

In [ ]:
# ProSST Single-Mutation Scoring
@torch.no_grad()
def score_single_variant(
    variant: object,
    ref_seq: str,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    ss_input_ids: torch.Tensor,
    offset: int,
    strict_wt_match: bool,
) -> tuple[float, bool, Optional[int]]:
    """
    Scores one single mutation as:

        log P(mutant residue) - log P(wild-type residue)

    at the masked mutation position.
    """
    muts = parse_variant(variant)

    if len(muts) == 0:
        return 0.0, True, None

    if len(muts) != 1:
        return float("nan"), False, None

    wt, pos1, mt = muts[0]

    pos_ref = pos1 + offset

    if pos_ref < 1 or pos_ref > len(ref_seq):
        return float("nan"), False, pos_ref

    wt_ref = ref_seq[pos_ref - 1].upper()
    wt_matches = wt_ref == wt

    if strict_wt_match and not wt_matches:
        return float("nan"), False, pos_ref

    # If strict=False and WT does not match,
    # use the residue from the PDB/reference sequence as WT.
    wt_used = wt if wt_matches else wt_ref

    # CLS token is at index 0, residue 1 is at token index 1
    token_index = pos_ref

    masked_input_ids = input_ids.clone()
    masked_input_ids[0, token_index] = prosst_tokenizer.mask_token_id

    outputs = prosst_model(
        input_ids=masked_input_ids,
        attention_mask=attention_mask,
        ss_input_ids=ss_input_ids,
    )

    log_probs = torch.log_softmax(
        outputs.logits[0, token_index],
        dim=-1,
    )

    wt_id = prosst_tokenizer.convert_tokens_to_ids(wt_used)
    mt_id = prosst_tokenizer.convert_tokens_to_ids(mt)

    score = log_probs[mt_id] - log_probs[wt_id]

    return float(score.item()), bool(wt_matches), int(pos_ref)

In [ ]:
def calculate_spearman_results(
    df: pd.DataFrame,
    score_cols: list[str],
    prediction_col: str = "prosst_delta_logprob",
) -> pd.DataFrame:
    """
    Calculates Spearman correlations between ProSST scores and experimental columns.
    """
    rows = []

    x = pd.to_numeric(df[prediction_col], errors="coerce")

    for col in score_cols:
        if col not in df.columns:
            rows.append(
                {
                    "score_col": col,
                    "n": 0,
                    "spearman_rho": float("nan"),
                    "p_value": float("nan"),
                    "note": "missing column",
                }
            )
            continue

        y = pd.to_numeric(df[col], errors="coerce")
        mask = x.notna() & y.notna()

        n = int(mask.sum())

        if n < 3:
            rows.append(
                {
                    "score_col": col,
                    "n": n,
                    "spearman_rho": float("nan"),
                    "p_value": float("nan"),
                    "note": "not enough valid pairs",
                }
            )
            continue

        rho, p = spearmanr(
            x[mask].values,
            y[mask].values,
        )

        rows.append(
            {
                "score_col": col,
                "n": n,
                "spearman_rho": float(rho),
                "p_value": float(p),
                "note": "",
            }
        )

    return pd.DataFrame(rows)

In [ ]:
def run_dataset(cfg: DatasetConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Runs the complete ProSST zero-shot scoring pipeline for one dataset.
    """
    print(f"\n==============================")
    print(f"Running {cfg.name}")
    print(f"==============================")

    df_raw = read_table(cfg)

    required_cols = [cfg.sequence_col, cfg.variant_col]
    missing_cols = [col for col in required_cols if col not in df_raw.columns]

    if missing_cols:
        raise ValueError(
            f"{cfg.name}: missing required columns: {missing_cols}"
        )

    df = df_raw.copy()

    # Optional: use num_mutations column if available/required
    if cfg.require_num_mutations_col:
        if "num_mutations" not in df.columns:
            raise ValueError(
                f"{cfg.name}: 'num_mutations' column is required but missing."
            )

        n_before_num = len(df)
        df = df.loc[df["num_mutations"] == 1].copy()
        print(f"num_mutations == 1: {len(df)}/{n_before_num}")

    # General single-mutation filter based on variant string
    df["prosst_is_single_mutation"] = df[cfg.variant_col].apply(is_single_mutation)

    n_before = len(df)
    df = df.loc[df["prosst_is_single_mutation"]].reset_index(drop=True)
    n_after = len(df)

    print(f"Valid single mutations: {n_after}/{n_before}")

    if len(df) == 0:
        raise ValueError(
            f"{cfg.name}: no single-mutation rows left after filtering."
        )

    wt_seq = str(df[cfg.sequence_col].iloc[0]).strip().upper()

    pdb_seq, structure_sequence = get_structure_from_pdb(cfg.pdb_path)

    validate_sequence_length(
        wt_seq=wt_seq,
        pdb_seq=pdb_seq,
        cfg=cfg,
    )

    # ProSST reference sequence comes from the PDB
    ref_seq = pdb_seq

    input_ids, attention_mask, ss_input_ids = tokenize_reference_sequence(
        ref_seq=ref_seq,
        structure_sequence=structure_sequence,
    )

    best_offset, hits, total = find_best_offset(
        df=df,
        ref_seq=ref_seq,
        variant_col=cfg.variant_col,
        search=cfg.offset_search,
    )

    match_rate = hits / max(total, 1)

    print(
        f"Offset used: {best_offset} | "
        f"WT matches: {hits}/{total} ({match_rate:.1%})"
    )

    scores = []
    wt_matches = []
    ref_positions = []

    for variant in tqdm(
        df[cfg.variant_col].tolist(),
        desc=f"Scoring {cfg.name}",
        unit="variant",
    ):
        score, matched, pos_ref = score_single_variant(
            variant=variant,
            ref_seq=ref_seq,
            input_ids=input_ids,
            attention_mask=attention_mask,
            ss_input_ids=ss_input_ids,
            offset=best_offset,
            strict_wt_match=cfg.strict_wt_match,
        )

        scores.append(score)
        wt_matches.append(matched)
        ref_positions.append(pos_ref)

    df_out = df.copy()

    df_out["prosst"] = scores
    df_out["prosst_delta_logprob"] = scores
    df_out["prosst_reference_sequence"] = "pdb_aa_seq"
    df_out["prosst_pdb_path"] = str(cfg.pdb_path)
    df_out["prosst_offset_used"] = best_offset
    df_out["prosst_pos_in_ref_1based"] = ref_positions
    df_out["prosst_wt_match_after_offset"] = wt_matches

    save_table(df_out, cfg.out_path)

    print(f"Saved results: {cfg.out_path}")

    metrics = calculate_spearman_results(
        df=df_out,
        score_cols=cfg.score_cols,
        prediction_col="prosst_delta_logprob",
    )

    metrics.insert(0, "dataset", cfg.name)
    metrics.insert(1, "n_rows_scored", len(df_out))
    metrics.insert(2, "offset_used", best_offset)
    metrics.insert(3, "wt_match_rate", match_rate)

    print("\nSpearman results:")
    print(metrics.to_string(index=False))

    return df_out, metrics

In [ ]:
# Run All Datasets

all_outputs = {}
all_metrics = []

for cfg in DATASETS:
    df_out, metrics = run_dataset(cfg)

    all_outputs[cfg.name] = df_out
    all_metrics.append(metrics)

summary = pd.concat(all_metrics, ignore_index=True)

summary_path = OUT_DIR / "prosst_spearman_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"\nSaved summary: {summary_path}")

summary


Running UBE4B
num_mutations == 1: 940/88375
Valid single mutations: 940/940
---------- Building Subgraphs ----------


100%|██████████| 1/1 [00:01<00:00,  1.84s/it]

Offset used: 1 | WT matches: 940/940 (100.0%)


Scoring UBE4B:   0%|          | 0/940 [00:00<?, ?variant/s]

Saved results: /home/kebau102/Masterarbeit/out/prosst/UBE4B_prosst.tsv

Spearman results:
dataset  n_rows_scored  offset_used  wt_match_rate score_col   n  spearman_rho      p_value note
  UBE4B            940            1            1.0     score 940      0.543991 1.629177e-73     

Running GRB2
num_mutations == 1: 1034/63366
Valid single mutations: 1034/1034
---------- Building Subgraphs ----------


...

100%|██████████| 1/1 [00:00<00:00,  1.18it/s]

Offset used: 0 | WT matches: 1558/1558 (100.0%)


Scoring PETase_WT_3:   0%|          | 0/1558 [00:00<?, ?variant/s]

Saved results: /home/kebau102/Masterarbeit/out/prosst/PETase_WT_3_prosst.csv

Spearman results:
    dataset  n_rows_scored  offset_used  wt_match_rate score_col  n  spearman_rho  p_value           note
PETase_WT_3           1558            0            1.0     score  0           NaN      NaN missing column

Saved summary: /home/kebau102/Masterarbeit/out/prosst/prosst_spearman_summary.csv


,dataset,n_rows_scored,offset_used,wt_match_rate,score_col,n,spearman_rho,p_value,note
0,UBE4B,940,1,1.0,score,940,0.543991,1.629177e-73,
1,GRB2,1034,1,1.0,score,1034,0.803044,3.048339e-234,
2,PTEN_activity,6564,1,1.0,score,6564,0.532497,0.000000e+00,
3,PTEN_abundance,4387,1,1.0,score,4387,0.477066,3.205534e-248,
4,Alpha-Amylase,8075,0,1.0,thermostability,7574,0.178331,3.711513e-55,
5,Alpha-Amylase,8075,0,1.0,expression,7575,0.373369,3.207948e-249,
6,Alpha-Amylase,8075,0,1.0,specific activity,7575,0.201384,3.759024e-70,
7,PETase_WT_1,1558,0,1.0,score,0,NaN,NaN,missing column
8,PETase_WT_2,1558,0,1.0,score,0,NaN,NaN,missing column
9,PETase_WT_3,1558,0,1.0,score,0,NaN,NaN,missing column
